### Step 1: Import API keys and libraries

In [18]:
import os
import json
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display


load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API_KEY is missing.")

### Step 2: Simple RAG

In [ ]:
from pathlib import Path
import chromadb
from document_chunker import DocumentChunker

# 1. Load the documents (every .txt sitting next to this script)
filenames = sorted(Path(".").glob("*.txt"))
docs = [(p.read_text(), {"source": p.name}) for p in filenames]

# 2. Chunk them all into one flat list, metadata preserved per chunk
chunker = DocumentChunker(chunk_size=800, chunk_overlap=100)
all_chunks = chunker.split_many(docs)

# 3. Store in Chroma (lets Chroma embed with its default model)
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="documents")

33 chunks


Chunk(text='You are allowed to send messages to the user via a notification tool. ', index=0, start_char=0, end_char=70, metadata={'source': 'document.txt', 'chunk_index': 0})

In [20]:
def display_chunks_md(chunks, n=None):
    """Render chunks as markdown. If n is given, only show the first n."""
    subset = chunks[:n] if n is not None else chunks
    for c in subset:
        md = (
            f"### Chunk {c.index}  \n"
            f"*chars {c.start_char}–{c.end_char} · ~{c.token_estimate} tokens*  \n"
            f"*metadata:* `{c.metadata}`\n\n"
            f"> {c.text.strip().replace(chr(10), chr(10) + '> ')}\n\n"
            f"---"
        )
        display(Markdown(md))
    if n is not None and n < len(chunks):
        display(Markdown(f"*...{len(chunks) - n} more chunks not shown*"))

# display_chunks_md(chunks, n=5)
from pprint import pprint
pprint(chunks)


[Chunk(text='You are allowed to send messages to the user via a notification '
            'tool. ',
       index=0,
       start_char=0,
       end_char=70,
       metadata={'chunk_index': 0, 'source': 'document.txt'}),
 Chunk(text=' notification tool.  You can use this tool to send messages that '
            'you want the user to see immediately, even if they are not '
            'currently looking at the chat interface. ',
       index=1,
       start_char=50,
       end_char=215,
       metadata={'chunk_index': 1, 'source': 'document.txt'}),
 Chunk(text='the chat interface.  You can use this tool to share important '
            'information, ask questions, or provide updates.\n'
            '\n'
            'Important: do not make things up. ',
       index=2,
       start_char=195,
       end_char=340,
       metadata={'chunk_index': 2, 'source': 'document.txt'}),
 Chunk(text="not make things up.  If you don't know the answer to a question, "
            'say "I don\'t know" or

In [ ]:
# Assign deterministic UUIDs to each chunk based on source document and chunk index, so we can upsert without creating duplicates
import uuid

### You can change the namespace to any fixed UUID, but if you do, you'll need to clear the collection to avoid duplicates, since the same chunk will get a different UUID with a different namespace. So we use a fixed namespace and keep it constant to allow upserts without duplicates.  See the following quoted code for how to mint a namespace if you want to use a different one.

"""
# Run this ONCE, in a throwaway cell, to mint your namespace:
import uuid
print(uuid.uuid4())
"""

NAMESPACE = uuid.NAMESPACE_DNS   # any fixed namespace works; just keep it constant

collection.upsert(
    ids=[
        str(uuid.uuid5(NAMESPACE, f"{c.metadata['source']}::{c.metadata['chunk_index']}"))
        for c in all_chunks
    ],
    documents=[c.text for c in all_chunks],
    metadatas=[c.metadata for c in all_chunks],
)

print(f"{len(all_chunks)} chunks from {len(filenames)} documents")
print(f"collection now holds {collection.count()} chunks")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

# ---- adjustable knob ----
n_clusters = 3          # <- change this to set the number of clusters
# -------------------------

# 1. Pull embeddings out of Chroma (must request them explicitly)
data = collection.get(include=["embeddings", "metadatas"])
embeddings = np.array(data["embeddings"])           # (70, 384)
sources = [m["source"] for m in data["metadatas"]]

# 2. KMeans on the ORIGINAL high-dim embeddings
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
labels = kmeans.fit_predict(embeddings)

# 3. t-SNE down to 2-D purely for plotting
coords = TSNE(
    n_components=2,
    perplexity=15,        # must be < number of points
    random_state=42,
    init="pca",
).fit_transform(embeddings)

# 4. Scatter, colored by KMeans cluster
plt.figure(figsize=(9, 7))
scatter = plt.scatter(coords[:, 0], coords[:, 1],
                      c=labels, cmap="tab10", alpha=0.8, s=60)
plt.legend(*scatter.legend_elements(), title="Cluster", fontsize=8)
plt.title(f"Chunk embeddings — t-SNE layout, KMeans (k={n_clusters})")
plt.xlabel("t-SNE 1"); plt.ylabel("t-SNE 2")
plt.tight_layout()
#plt.show()

# Plotly hover labels
import plotly.express as px

fig = px.scatter(
    x=coords[:, 0], y=coords[:, 1],
    color=labels.astype(str),
    hover_data={
        "id": data["ids"],
        "chunk_index": [m["chunk_index"] for m in data["metadatas"]],
        "source": [m["source"] for m in data["metadatas"]],
    },
    title="Chunk embeddings (hover for id)",
)
fig.update_traces(marker=dict(size=9, opacity=0.8))
fig.show()

# Describe each cluster by the most common source document among its chunks using a prompt to GPT-4.1-mini
data = collection.get(include=["embeddings", "documents", "metadatas"])
embeddings = np.array(data["embeddings"])
docs = data["documents"]
labels = kmeans.fit_predict(embeddings) 


client = OpenAI(api_key=OPENAI_API_KEY)

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

k = labels.max() + 1

# Pool each cluster's chunks into one "super-document"
for c in range(k):
    sample = [d for d, lab in zip(docs, labels) if lab == c][:5]
    prompt = ("Give a short 5–8 word label for the common theme of these excerpts:\n\n"
              + "\n---\n".join(sample))
    resp = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
    )
    print(f"Cluster {c}: {resp.choices[0].message.content}")

In [26]:
tools = []

In [27]:
pushover_user = os.getenv("PUSHOEVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

import requests 
def send_notification(message: str):
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)
    
send_notification_function = {
    "name": "send_notification",
    "description": "Send a notification to the real-world version of you via Pushover.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The message to send as a notification."
            }
        },
        "required": ["message"]
    }
}

tools.append({"type": "function", "function":send_notification_function})

In [28]:
import random

def dice_roll():
    result = random.randint(1, 6)
    return result

roll_dice_function = {
    "name": "dice_roll",
    "description": "Simulates rolling a single six-sided die and returns the result.  Use this when the user wants to roll a die for games, dcisions, or randcom numbers.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": []
    }
}

tools.append({"type": "function", "function": roll_dice_function})

In [29]:
def handle_tool_call(tool_calls):
    tool_results = []
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        
        if function_name == "send_notification":
            send_notification(args["message"])
            content = f"Notification sent with message: {args['message']}"
        elif function_name == "dice_roll":
            content = f"Die rolled and the result is: {dice_roll()}"
        else:
            content = f"Unknown tool: {function_name}"
        
        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id
        }
        
        tool_results.append(tool_call_result)
        
    return tool_results

### Step 6: Test Retrieval

In [30]:
test_query = "days off"
test_query2 = "teamwork and collaboration"

response = client.embeddings.create(
    model = "text-embedding-3-small",
    input = [test_query, test_query2]
)

query_embedding = [response.data[0].embedding, response.data[1].embedding]

# Search chromadb
results = collection.query(
    query_embeddings=query_embedding,
    n_results=3
)

print(f"Query: {test_query}")
print("Retrieved Chunks:")
for a, b in zip(results["documents"], results["metadatas"][0]):
    print(f"Chunk {b['chunk_index']}:\n{a}\n")


Query: days off
Retrieved Chunks:
Chunk 24:
['s and watch movies.  By working at a video store, he was allowed to watch screeners of new movies a week before they were released to the public for rent.  His friends were Phillip and Derrek. ', ' data-focused work.\nOver the past year, Jared has completed two intensive DataTalks.Club programs back-to-back: the Machine Learning Zoomcamp (2026) and the Data Engineering Zoomcamp (2026). ', 'Phillip and Derrek.  He also enjoyed working out at the YMCA and playing basketball.\n\n']

Chunk 12:
['mework submissions, and earlier project work from boot.dev and Nucamp. He is particularly drawn to research data science roles where rigorous quantitative thinking applies to large-scale measurement, ', ' data-focused work.\nOver the past year, Jared has completed two intensive DataTalks.Club programs back-to-back: the Machine Learning Zoomcamp (2026) and the Data Engineering Zoomcamp (2026). ', 'the chat interface.  You can use this tool to share impor

### Step 7: Function to Process the Conversation Turn (with RAG)

In [40]:
system_message = "You are Jared Gavin and you always respond in a Captain Jack Sparrow style but you are NOT captain jack sparrow."

In [ ]:
dclient = OpenAI(api_key=OPENAI_API_KEY)   # define once, outside the function

def respond_ai(message, history):
    # RAG retrieval — Chroma embeds the query with the same default model
    results = collection.query(
        query_texts=[message],
        n_results=3,
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]
    context = "\n---\n".join(
        f"[{m['source']}] {d}" for d, m in zip(docs, metas)
    )

    system_message_enhanced = system_message + "\n\nContext:\n" + context

    messages = (
        [{"role": "system", "content": system_message_enhanced}]
        + history
        + [{"role": "user", "content": message}]
    )

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=tools,
        
    )

    return response.choices[0].message.content
    
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
    )
    
    reply = response.choices[0].message.content
    return reply

In [42]:
gr.ChatInterface(fn=respond_ai).launch(inbrowser=True, inline=False)

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.
